# EDA DSA Docs

Exploratory analysis for DSA documentation corpus.

Steps:
- Inventory DSA categories and file counts.
- Check embeddings cache if present.
- Run the training data audit and summarize outputs.



In [ ]:
from __future__ import annotations

import json
import os
import sys
import subprocess
from pathlib import Path

# Resolve repo root from the notebook location.
REPO_ROOT = Path.cwd()
for parent in [REPO_ROOT] + list(REPO_ROOT.parents):
    if (parent / 'scripts').exists() and (parent / 'notebooks').exists():
        REPO_ROOT = parent
        break

# Ensure local modules are importable.
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / 'src'))

PY = sys.executable

def run(cmd: list[str]) -> None:
    # Run a command from the repo root with PYTHONPATH set.
    env = os.environ.copy()
    env['PYTHONPATH'] = os.pathsep.join([str(REPO_ROOT / 'src'), str(REPO_ROOT)])
    print('$', ' '.join(cmd))
    subprocess.run(cmd, cwd=str(REPO_ROOT), check=True, env=env)

def show_json(rel_path: str) -> None:
    path = REPO_ROOT / rel_path
    if not path.exists():
        print('Missing:', path)
        return
    try:
        data = json.loads(path.read_text(encoding='utf-8'))
    except Exception:
        print(path.read_text(encoding='utf-8', errors='ignore')[:2000])
        return
    print(json.dumps(data, indent=2))

def list_dir(rel_path: str, limit: int = 20) -> None:
    path = REPO_ROOT / rel_path
    if not path.exists():
        print('Missing:', path)
        return
    print(f'\n{rel_path}/')
    for item in sorted(path.iterdir())[:limit]:
        print(' -', item.name)


In [ ]:
from pathlib import Path

base_dir = REPO_ROOT / 'data'
dsa_docs = base_dir / 'dsa_docs'
embeddings_dir = base_dir / 'dsa_embeddings'

summary = {
    'dsa_docs': {},
    'dsa_embeddings': {},
}

print('DSA docs:', dsa_docs)
if dsa_docs.exists():
    categories = [d for d in sorted(dsa_docs.iterdir()) if d.is_dir()]
    summary['dsa_docs']['category_count'] = len(categories)
    for cat in categories:
        files = [p for p in cat.rglob('*') if p.is_file()]
        summary['dsa_docs'][cat.name] = len(files)
    for cat, count in list(summary['dsa_docs'].items())[:10]:
        if cat == 'category_count':
            continue
        print(f'{cat}: {count} files')
else:
    print('Missing:', dsa_docs)

print('Embeddings:', embeddings_dir)
if embeddings_dir.exists():
    files = [p for p in embeddings_dir.rglob('*') if p.is_file()]
    summary['dsa_embeddings']['file_count'] = len(files)
    for sample in files[:8]:
        print(' -', sample.relative_to(REPO_ROOT))
else:
    print('Missing:', embeddings_dir)


In [ ]:
# Preview a sample DSA document if available.
if dsa_docs.exists():
    sample_files = [p for p in dsa_docs.rglob('*') if p.is_file()]
    if sample_files:
        sample = sample_files[0]
        print('Preview:', sample.relative_to(REPO_ROOT))
        print(sample.read_text(encoding='utf-8', errors='ignore')[:2000])
    else:
        print('No DSA docs found for preview.')


In [ ]:
# Persist summary for quick reference.
report_dir = REPO_ROOT / 'reports'
report_dir.mkdir(parents=True, exist_ok=True)
summary_path = report_dir / 'eda_dsa_docs_summary.json'
summary_path.write_text(json.dumps(summary, indent=2))
print('Saved summary to', summary_path)


In [ ]:
# Generate a training data audit
run([PY, 'scripts/training_data_audit.py'])



In [ ]:
# Summarize dsa-related entries from the training data audit.
audit_path = REPO_ROOT / 'reports' / 'TRAINING_DATA.json'
if not audit_path.exists():
    print('Missing:', audit_path)
else:
    audit = json.loads(audit_path.read_text(encoding='utf-8'))
    items = [
        item for item in audit.get('required', []) + audit.get('optional', [])
        if 'dsa' in str(item.get('name', '')).lower()
    ]
    if not items:
        print('No dsa entries found in TRAINING_DATA.json')
    else:
        print('dsa datasets in audit:')
        for item in items:
            print(' -', item.get('name'), '|', item.get('status'), '|', item.get('path'))


In [ ]:
# Quick artifact index for verification.
for folder in ['models', 'experiments', 'artifacts', 'runs', 'reports', 'logs']:
    path = REPO_ROOT / folder
    if not path.exists():
        continue
    print(f'\n{folder}/')
    for item in sorted(path.iterdir())[:20]:
        print(' -', item.name)
